# Entanglement and Superposition

This notebook provides an in-depth exploration of two of the most fundamental and counter-intuitive phenomena in quantum mechanics: **superposition** and **entanglement**. These quantum features are at the heart of quantum computing's potential advantages over classical computing.

## Learning Objectives

By the end of this notebook, you will understand:

1. **Quantum Superposition** - The principle of superposition and its implications
2. **Entanglement** - Non-classical correlations between quantum systems
3. **Bell States** - The maximally entangled two-qubit states
4. **Quantum Correlations** - How entanglement leads to stronger-than-classical correlations
5. **Applications** - How superposition and entanglement enable quantum algorithms

## Prerequisites

- Completion of quantum_computing_basics.ipynb
- Basic understanding of linear algebra
- Familiarity with Qiskit basics

---

## 1. Introduction: The Quantum World

Classical physics assumes objects have definite properties at all times. A coin is either heads or tails, a switch is either on or off. Quantum mechanics challenges this assumption by showing that particles can exist in **superpositions** of multiple states simultaneously.

Entanglement goes even further, showing that two particles can be correlated in ways that have no classical explanation. This notebook explores these phenomena in detail.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, Math, Latex

# Import Qiskit components
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator, DensityMatrix
from qiskit.visualization import plot_bloch_vector, plot_histogram
from qiskit_aer import AerSimulator

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)

print("✓ Environment ready for exploring entanglement and superposition!")

## 2. Quantum Superposition Deep Dive

### 2.1 The Principle of Superposition

In quantum mechanics, any valid state can be expressed as a linear combination of basis states:

$$|\psi\rangle = \sum_i c_i |i\rangle$$

where the coefficients c_i are complex amplitudes satisfying Σ|c_i|² = 1.

**Key insight**: The superposition is not a mere probability distribution - it's an actual physical state. The amplitudes interfere with each other, leading to quantum phenomena that have no classical analogue.

In [ ]:
# Explore superposition states

print("=== Superposition States ===\n")

# Define basis states
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Create various superposition states
states = {
    '|0⟩': ket_0,
    '|1⟩': ket_1,
    '|+⟩': (ket_0 + ket_1) / np.sqrt(2),
    '|-⟩': (ket_0 - ket_1) / np.sqrt(2),
    '|+i⟩': (ket_0 + 1j*ket_1) / np.sqrt(2),
    '|-i⟩': (ket_0 - 1j*ket_1) / np.sqrt(2),
}

print("State amplitudes and probabilities:")
for name, state in states.items():
    prob_0 = np.abs(state[0])**2
    prob_1 = np.abs(state[1])**2
    print(f"{name}: α={state[0]:.3f}, β={state[1]:.3f} → P(0)={prob_0:.2f}, P(1)={prob_1:.2f}")

# Visualize on Bloch sphere
fig = plt.figure(figsize=(15, 10))

for i, (name, state) in enumerate(states.items()):
    ax = fig.add_subplot(2, 3, i+1, projection='3d')
    
    # Calculate Bloch sphere coordinates
    x = 2 * np.real(state[0].conj() * state[1])
    y = 2 * np.imag(state[0].conj() * state[1])
    z = np.abs(state[0])**2 - np.abs(state[1])**2
    
    ax.quiver(0, 0, 0, x, y, z, length=0.8, color='red', arrow_length_ratio=0.3)
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([-1, 1])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(name)

plt.suptitle('Superposition States on the Bloch Sphere', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Phase and Interference

The phase of a quantum state is crucial. Two states with the same probabilities can have different phases, leading to different interference patterns.

The relative phase between |0⟩ and |1⟩ components affects:
- The position on the equator of the Bloch sphere
- Interference patterns in quantum circuits
- The outcome of measurements in different bases

In [ ]:
# Explore phase effects

print("=== Phase Effects in Superposition ===\n")

# Create states with different phases
phases = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 5*np.pi/4, 3*np.pi/2, 7*np.pi/4]
phase_states = {}

for phase in phases:
    state = (ket_0 + np.exp(1j*phase)*ket_1) / np.sqrt(2)
    phase_states[f'φ={phase/np.pi:.2f}π'] = state

print("States with different phases (equal superposition):")
for name, state in phase_states.items():
    # Calculate Bloch coordinates
    x = 2 * np.real(state[0].conj() * state[1])
    y = 2 * np.imag(state[0].conj() * state[1])
    z = 0  # Equal superposition has z=0
    print(f"{name}: (x, y, z) = ({x:.3f}, {y:.3f}, {z:.3f})")

# Visualize phase on unit circle
fig, ax = plt.subplots(figsize=(10, 10))

theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='Unit circle')

for phase in phases:
    x = np.cos(phase)
    y = np.sin(phase)
    ax.scatter(x, y, s=100, zorder=5)
    ax.annotate(f'φ={phase/np.pi:.2f}π', (x+0.1, y+0.1), fontsize=10)

ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-1.5, 1.5])
ax.set_xlabel('Re(e^{iφ})')
ax.set_ylabel('Im(e^{iφ})')
ax.set_title('Phase Visualization on Unit Circle', fontsize=14)
ax.set_aspect('equal')
ax.grid(True)
plt.show()

### 2.3 Quantum Interference

Quantum interference is the heart of many quantum algorithms. By carefully constructing interference patterns, we can amplify correct answers and cancel out wrong ones.

The probability of measuring a state |x⟩ is:

$$P(x) = |\langle x|\psi\rangle|^2 = |\sum_i c_i \langle x|i\rangle|^2$$

The cross terms (i ≠ j) in |Σ c_i|² are responsible for interference.

In [ ]:
# Demonstrate quantum interference

print("=== Quantum Interference Demonstration ===\n")

# Create a circuit that demonstrates interference
from qiskit import QuantumCircuit

# Circuit 1: H on |0⟩ then measure in X basis
qc1 = QuantumCircuit(1, 1)
qc1.h(0)  # Create |+⟩
qc1.h(0)  # Interfere (should give |0⟩ with high probability)
qc1.measure(0, 0)

print("Circuit: H-H on |0⟩")
print(qc1.draw())

# Run on simulator
simulator = AerSimulator()
result1 = simulator.run(qc1, shots=1000).result()
counts1 = result1.get_counts()
print(f"Results: {counts1}")

# Circuit 2: H on |0⟩ then X then H
qc2 = QuantumCircuit(1, 1)
qc2.h(0)   # |+⟩
qc2.x(0)   # Flip to |−⟩
qc2.h(0)   # Interfere
qc2.measure(0, 0)

print("\nCircuit: H-X-H on |0⟩")
print(qc2.draw())

result2 = simulator.run(qc2, shots=1000).result()
counts2 = result2.get_counts()
print(f"Results: {counts2}")

# Circuit 3: H on |0⟩ then Z then H
qc3 = QuantumCircuit(1, 1)
qc3.h(0)   # |+⟩
qc3.z(0)   # Apply phase
qc3.h(0)   # Interfere
qc3.measure(0, 0)

print("\nCircuit: H-Z-H on |0⟩")
print(qc3.draw())

result3 = simulator.run(qc3, shots=1000).result()
counts3 = result3.get_counts()
print(f"Results: {counts3}")

print("\n=== Analysis ===")
print("H-H: Constructive interference → |0⟩")
print("H-X-H: Destructive interference → |1⟩")
print("H-Z-H: Different phase → back to |0⟩")

## 3. Entanglement Deep Dive

### 3.1 What is Entanglement?

Entanglement is a quantum phenomenon where two or more particles become correlated in a way that cannot be explained by classical physics. When particles are entangled, measuring one particle instantly affects the state of the other, regardless of the distance between them.

**Einstein's famous concern**: Einstein called this "spooky action at a distance" and it troubled him deeply. However, countless experiments have confirmed that entanglement is real and violates classical expectations.

In [ ]:
# Explore entangled states

print("=== Creating Entangled States ===\n")

# Create Bell states (maximally entangled)
bell_states = {
    '|Φ⁺⟩': (np.array([1, 0, 0, 1], dtype=complex)) / np.sqrt(2),  # (|00⟩ + |11⟩)/√2
    '|Φ⁻⟩': (np.array([1, 0, 0, -1], dtype=complex)) / np.sqrt(2), # (|00⟩ - |11⟩)/√2
    '|Ψ⁺⟩': (np.array([0, 1, 1, 0], dtype=complex)) / np.sqrt(2),  # (|01⟩ + |10⟩)/√2
    '|Ψ⁻⟩': (np.array([0, 1, -1, 0], dtype=complex)) / np.sqrt(2), # (|01⟩ - |10⟩)/√2
}

print("The Four Bell States:")
for name, state in bell_states.items():
    print(f"\n{name}:")
    print(f"  Vector: {state}")
    
    # Show as superposition
    print(f"  = (|00⟩ + {'-' if state[3] < 0 else '+'}|11⟩)/√2")

# Create a circuit to generate Bell state |Φ⁺⟩
bell_circuit = QuantumCircuit(2, 2)
bell_circuit.h(0)
bell_circuit.cx(0, 1)

print("\n=== Bell State Circuit ===")
print(bell_circuit.draw())

bell_sv = Statevector.from_instruction(bell_circuit)
print(f"\nOutput state: {bell_sv}")

### 3.2 Detecting Entanglement

How do we know if a state is entangled? One method is to compute the **reduced density matrix** and check its purity.

For a separable state |ψ⟩ = |a⟩ ⊗ |b⟩, the reduced density matrix of either subsystem is pure (purity = 1).
For an entangled state, the reduced density matrices are mixed (purity < 1).

In [ ]:
# Detect entanglement via reduced density matrices

def compute_purity(rho):
    """Compute purity Tr(ρ²)"""
    return np.real(np.trace(rho @ rho))

def partial_trace(rho, sys_dim, trace_qubit):
    """Compute partial trace over one qubit"""
    d = sys_dim
    result = np.zeros((d, d), dtype=complex)
    
    if trace_qubit == 0:
        for i in range(d):
            for j in range(d):
                result[i, j] = rho[i*d + j, i*d + j] + rho[i*d + j + d, i*d + j + d]
    else:
        for i in range(d):
            for j in range(d):
                result[i, j] = rho[i, j] + rho[i+d, j+d]
    
    return result

print("=== Entanglement Detection ===\n")

# Test separable state |+⟩ ⊗ |0⟩
plus = (ket_0 + ket_1) / np.sqrt(2)
separable = np.kron(plus, ket_0)
sep_dm = np.outer(separable, separable.conj())
sep_rho_0 = partial_trace(sep_dm, 2, 0)
sep_rho_1 = partial_trace(sep_dm, 2, 1)

print("Separable state |+⟩ ⊗ |0⟩:")
print(f"  Purity of reduced ρ₀: {compute_purity(sep_rho_0):.4f}")
print(f"  Purity of reduced ρ₁: {compute_purity(sep_rho_1):.4f}")
print(f"  → Both pure → SEPARABLE")

# Test Bell state |Φ⁺⟩
bell = bell_states['|Φ⁺⟩']
bell_dm = np.outer(bell, bell.conj())
bell_rho_0 = partial_trace(bell_dm, 2, 0)
bell_rho_1 = partial_trace(bell_dm, 2, 1)

print("\nBell state |Φ⁺⟩:")
print(f"  Purity of reduced ρ₀: {compute_purity(bell_rho_0):.4f}")
print(f"  Purity of reduced ρ₁: {compute_purity(bell_rho_1):.4f}")
print(f"  → Both mixed → ENTANGLED")

# Show reduced density matrices
print("\n=== Reduced Density Matrices ===")
print("Bell state reduced ρ₀:")
print(bell_rho_0)
print("\nThis is the maximally mixed state I/2!")

### 3.3 Bell Inequality and Non-Locality

The Bell inequality provides a way to experimentally distinguish quantum entanglement from classical correlations. Violation of a Bell inequality proves that no classical theory can reproduce quantum predictions.

**CHSH Inequality**: For any classical (local hidden variable) theory:

$$|E(a,b) - E(a,b') + E(a',b) + E(a',b')| \leq 2$$

Quantum mechanics can achieve up to 2√2 ≈ 2.83 (Tsirelson bound).

In [ ]:
# Demonstrate Bell inequality violation

print("=== Bell Inequality (CHSH) Demonstration ===\n")

# Define measurement operators for different bases
# X and Z bases
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

def compute_correlation(state, measureA, measureB):
    """Compute correlation E(a,b) = ⟨AB⟩"""
    # For 2-qubit state, compute ⟨A⊗B⟩
    op = np.kron(measureA, measureB)
    return np.real(np.vdot(state, op @ state))

# Use Bell state |Φ⁺⟩
bell = bell_states['|Φ⁺⟩']

# Define measurement settings
A = X  # Alice's measurement at angle 0
A_prime = Z  # Alice's measurement at angle π/2
B = (X + Z) / np.sqrt(2)  # Bob's measurement at angle π/4
B_prime = (X - Z) / np.sqrt(2)  # Bob's measurement at angle -π/4

# Compute correlations
E_ab = compute_correlation(bell, A, B)
E_ab_prime = compute_correlation(bell, A, B_prime)
E_a_prime_b = compute_correlation(bell, A_prime, B)
E_a_prime_b_prime = compute_correlation(bell, A_prime, B_prime)

print("Correlation values:")
print(f"  E(a,b) = {E_ab:.4f}")
print(f"  E(a,b') = {E_ab_prime:.4f}")
print(f"  E(a',b) = {E_a_prime_b:.4f}")
print(f"  E(a',b') = {E_a_prime_b_prime:.4f}")

chsh = abs(E_ab - E_ab_prime + E_a_prime_b + E_a_prime_b_prime)
print(f"\nCHSH value: {chsh:.4f}")
print(f"Classical bound: ≤ 2")
print(f"Quantum bound: ≤ 2√2 ≈ 2.828")
print(f"\n→ CHSH = {chsh:.4f} > 2 → Bell inequality VIOLATED!")
print("This proves quantum entanglement is non-classical!")

## 4. Multi-Qubit Entanglement

### 4.1 GHZ States

The GHZ (Greenberger-Horne-Zeilinger) state is a generalisation of the Bell state to multiple qubits:

$$|GHZ_n\rangle = \frac{|0\rangle^{\otimes n} + |1\rangle^{\otimes n}}{\sqrt{2}}$$

This state exhibits genuine n-partite entanglement.

In [ ]:
# Create GHZ states

print("=== GHZ States ===\n")

for n_qubits in [3, 4, 5]:
    # Create GHZ circuit
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.h(0)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
    
    # Get statevector
    sv = Statevector.from_instruction(qc)
    print(f"{n_qubits}-qubit GHZ state:")
    print(f"  Circuit depth: {qc.depth()}")
    print(f"  Non-zero amplitudes: {np.sum(np.abs(sv.data) > 1e-10)}")
    print(f"  State: (|0...0⟩ + |1...1⟩)/√2\n")

# Test GHZ correlations
print("=== GHZ Correlations ===")

ghz3 = QuantumCircuit(3, 3)
ghz3.h(0)
ghz3.cx(0, 1)
ghz3.cx(1, 2)

sv3 = Statevector.from_instruction(ghz3)
print(f"\n3-qubit GHZ state: {sv3}")

# Run with measurement
ghz3_meas = QuantumCircuit(3, 3)
ghz3_meas.h(0)
ghz3_meas.cx(0, 1)
ghz3_meas.cx(1, 2)
ghz3_meas.measure([0, 1, 2], [0, 1, 2])

result = simulator.run(ghz3_meas, shots=1000).result()
counts = result.get_counts()
print(f"\nMeasurement results: {counts}")
print("Only |000⟩ and |111⟩ observed - perfect correlation!")

### 4.2 W States

The W state is another important 3-qubit entangled state:

$$|W\rangle = \frac{|001\rangle + |010\rangle + |100\rangle}{\sqrt{3}}$$

Unlike GHZ, the W state cannot be transformed into GHZ by local operations and classical communication (LOCC).

In [ ]:
# Create W state

print("=== W State ===\n")

# Create W state circuit
w_circuit = QuantumCircuit(3, 3)
w_circuit.h(0)
w_circuit.x(1)
w_circuit.ccx(0, 1, 2)
w_circuit.x(1)
w_circuit.h(0)
w_circuit.cx(0, 1)
w_circuit.x(1)
w_circuit.ccx(0, 1, 2)

print("W state circuit:")
print(w_circuit.draw())

w_sv = Statevector.from_instruction(w_circuit)
print(f"\nW state: {w_sv}")

# Verify it's the W state
w_expected = (np.array([0, 1, 1, 0, 1, 0, 0, 0], dtype=complex)) / np.sqrt(3)
print(f"Expected W state: {w_expected}")
print(f"Match? {np.allclose(w_sv.data, w_expected)}")

# Measure
w_meas = QuantumCircuit(3, 3)
w_meas.h(0)
w_meas.x(1)
w_meas.ccx(0, 1, 2)
w_meas.x(1)
w_meas.h(0)
w_meas.cx(0, 1)
w_meas.x(1)
w_meas.ccx(0, 1, 2)
w_meas.measure([0, 1, 2], [0, 1, 2])

result = simulator.run(w_meas, shots=1000).result()
counts = result.get_counts()
print(f"\nMeasurement results: {counts}")
print("Only states with exactly one |1⟩ observed!")

## 5. Applications in Quantum Computing

### 5.1 Superposition in Quantum Algorithms

Superposition is essential for many quantum algorithms:

- **Grover's algorithm**: Uses superposition to search unstructured databases
- **Quantum Fourier Transform**: Operates on superpositions
- **Variational algorithms**: Use superposition states as ansatzes

In [ ]:
# Demonstrate superposition in a simple quantum algorithm

print("=== Superposition in Quantum Search (Simplified) ===\n")

# Create a simple oracle for marking state |11⟩
oracle = QuantumCircuit(2, 2)
oracle.cz(0, 1)  # Phase oracle

print("Simple phase oracle:")
print(oracle.draw())

# Diffusion operator (amplitude amplification)
diffusion = QuantumCircuit(2, 2)
diffusion.h([0, 1])
diffusion.x([0, 1])
diffusion.cz(0, 1)
diffusion.x([0, 1])
diffusion.h([0, 1])

print("\nDiffusion operator:")
print(diffusion.draw())

# Full Grover iteration
grover = QuantumCircuit(2, 2)
grover.h([0, 1])  # Initial superposition
grover = grover.compose(oracle)
grover = grover.compose(diffusion)

print("\nOne Grover iteration:")
print(grover.draw())

# Get state
grover_sv = Statevector.from_instruction(grover)
print(f"\nOutput state: {grover_sv}")

# Measure
grover.measure([0, 1], [0, 1])
result = simulator.run(grover, shots=1000).result()
counts = result.get_counts()
print(f"\nMeasurement results: {counts}")
print("Amplitude increased for |11⟩ state!")

### 5.2 Entanglement in Quantum Algorithms

Entanglement is a key resource for:

- **Quantum teleportation**: Using entanglement to transmit quantum states
- **Quantum key distribution**: BB84 protocol uses entanglement
- **Variational quantum eigensolver (VQE)**: Entangled ansatzes are more powerful
- **Quantum error correction**: Requires entanglement for encoding

In [ ]:
# Demonstrate quantum teleportation (simplified)

print("=== Quantum Teleportation Circuit ===\n")

# Teleportation circuit
# Alice has qubit 0 (unknown state), qubit 1 (entangled with Bob)
# Bob has qubit 2
teleport = QuantumCircuit(3, 2)

# Step 1: Prepare unknown state on qubit 0
teleport.h(0)  # Create superposition state to teleport

# Step 2: Create entanglement between qubit 1 and 2
teleport.h(1)
teleport.cx(1, 2)

# Step 3: Bell measurement on qubits 0 and 1
teleport.cx(0, 1)
teleport.h(0)

# Step 4: Classical communication and correction
teleport.measure([0, 1], [0, 1])

# Step 5: Apply corrections based on measurement
teleport.x(2).c_ife(0, 1)  # If first bit is 1, apply X
teleport.z(2).c_ife(0, 0)  # If second bit is 1, apply Z

print(teleport.draw())

print("\nNote: This circuit demonstrates the teleportation protocol.")
print("The unknown state on qubit 0 is transferred to qubit 2.")

## 6. Summary and Key Concepts

Let's review what we've learned about superposition and entanglement:

In [ ]:
# Summary

print("="*60)
print("ENTANGLEMENT AND SUPERPOSITION - KEY CONCEPTS")
print("="*60)

summary = """
1. SUPERPOSITION
   - Qubits can exist in multiple states simultaneously
   - State: |ψ⟩ = α|0⟩ + β|1⟩ with |α|² + |β|² = 1
   - Phase affects interference patterns
   - Essential for quantum parallelism

2. ENTANGLEMENT
   - Non-classical correlations between qubits
   - Bell states: (|00⟩ ± |11⟩)/√2, (|01⟩ ± |10⟩)/√2
   - Violates Bell inequalities (CHSH > 2)
   - Reduced density matrices are mixed

3. GHZ AND W STATES
   - GHZ: (|0...0⟩ + |1...1⟩)/√2 (maximally entangled)
   - W: (|001⟩ + |010⟩ + |100⟩)/√3
   - Not convertible via LOCC

4. APPLICATIONS
   - Grover's search uses superposition
   - Quantum teleportation uses entanglement
   - VQE uses entangled ansatzes
   - Quantum cryptography uses entanglement
"""
print(summary)

# Visualize entanglement spectrum
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

states_to_plot = [
    ("Separable |+⟩⊗|0⟩", "Green", ""),
    ("Partially Entangled", "Orange", ""),
    ("Bell State |Φ⁺⟩", "Red", "")
]

for ax, (title, color, _) in zip(axes, states_to_plot):
    ax.text(0.5, 0.5, title, ha='center', va='center', fontsize=14)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_facecolor({'Green': '#90EE90', 'Orange': '#FFD700', 'Red': '#FFB6C1'}[color])

plt.suptitle("Entanglement Spectrum", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Exercises

### Exercise 1: Create Superposition States
Create a circuit that produces the state (|0⟩ + i|1⟩)/√2. Verify using statevector simulation.

### Exercise 2: Bell State Measurement
Create a circuit that performs Bell basis measurement on two qubits.

### Exercise 3: Entanglement Verification
Write code to verify that a given 2-qubit state is entangled using the partial trace method.

### Exercise 4: GHZ Correlations
Create a 4-qubit GHZ state and verify that measuring all qubits in the X basis gives correlated results.

### Exercise 5: Quantum Interference
Design a circuit that demonstrates quantum interference by constructing a state where certain outcomes are cancelled out.

### Exercise 6: W State Properties
Compute the reduced density matrices of the W state and compare with the GHZ state.

## 8. Further Reading and Resources

- **Michael Nielsen & Isaac Chuang**: "Quantum Computation and Quantum Information"
- **Qiskit Textbook**: https://qiskit.org/textbook/
- **IBM Quantum Experience**: https://quantum-computing.ibm.com/

---

## Next Steps

In the next notebook, we'll explore **Grover's Algorithm** - one of the most important quantum algorithms that demonstrates the power of superposition and entanglement for search problems.

---
*This notebook is part of the Quantum Machine Learning series. For more resources, visit the repository README.*